In [ ]:
#export

"""
GUI de Incidencias + IA (Tkinter)
---------------------------------
Requisitos:
- Python 3.9+
- Archivos en el mismo entorno: `ollama_manager.py` e `Incidencias.py` (o incidencias.py)

Funciones clave:
- Selección de modelos (general e incidencia)
- Selección de aplicación
- Cargar Excel de incidencias
- Ejecutar pipeline único: Incidencias.main_incidencias(url, fichero, lista_campos, ...)
- Logging en vivo (thread-safe y multiproceso) en la ventana
- Editor de settings.ini
- Cierre que mata el subproceso del pipeline

Variables globales pedidas:
- modelos_general
- modelos_incidencia
- fichero_origen_path
"""
from __future__ import annotations

import os 
import sys 
import threading 
import logging 
import logging.handlers 
import queue 
import inspect 
import tkinter as tk 
from tkinter import ttk, filedialog, messagebox 
import configparser 
from pathlib import Path

# ---------------------------
# Versión de la aplicación
# ---------------------------
__version__ = "0.3.0"

# ---------------------------
# Variables globales pedidas
# ---------------------------
modelos_general: list[str] = []
modelos_incidencia: list[str] = []
fichero_origen_path: str | None = None

# ---------------------------
# Importación de módulos del proyecto
# ---------------------------
try:
    import ollama_manager as om
except Exception as e:
    om = None
    print("[ADVERTENCIA] No se pudo importar ollama_manager.py:", e)

try:
    import incidencias as inc  # nombre de módulo en minúsculas por compatibilidad
except Exception as e:
    inc = None
    print("[ADVERTENCIA] No se pudo importar Incidencias.py/incidencias.py:", e)

# ---------------------------
# Rutas y settings
# ---------------------------
def _resolve_app_dir() -> Path:
    """Carpeta base robusta:
    - Script .py: __file__
    - Binario (pyinstaller): sys.executable
    - Jupyter/REPL: Path.cwd()
    """
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

APP_DIR = _resolve_app_dir()
SETTINGS_PATH = APP_DIR / "settings.ini"


def cargar_settings() -> configparser.ConfigParser:
    cfg = configparser.ConfigParser()
    if not SETTINGS_PATH.exists():
        cfg["DEFAULT"] = {
            "url_ia": "http://localhost:11434",
            "path_salida": str(APP_DIR),
            # Ahora prompt1_defecto puede ser TEXTO directo o RUTA a un archivo
            "prompt1_defecto": "prompt1_defecto.txt",
            "prompt2_defecto": "prompt2_defecto.txt",
            # Campos por defecto del Excel (coinciden con tu especificación inicial):
            "lista_campos": "Número, Descripción, Notas de resolución",
            "thinking_incidencia": "True",
            "thinking_general": "True",
            "temperature": "0.7",
        }
        with open(SETTINGS_PATH, "w", encoding="utf-8") as f:
            cfg.write(f)
    else:
        cfg.read(SETTINGS_PATH, encoding="utf-8")
    return cfg

# ---------------------------
# Worker de subproceso (pipeline)
# ---------------------------

'''
def _pipeline_worker(mp_log_queue: multiprocessing.Queue,
                     app: str,
                     fichero_path: str,
                     modelos_inci: list[str],
                     modelos_gen: list[str]) -> None:
    """Subproceso: corre main_incidencias y manda logs a la cola multiproceso."""
    # Redirigir logging del hijo a la cola
    root = logging.getLogger()
    for h in list(root.handlers):
        root.removeHandler(h)
    qh = logging.handlers.QueueHandler(mp_log_queue)
    root.setLevel(logging.INFO)
    root.addHandler(qh)
    ui_fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", "%Y-%m-%d %H:%M:%S")

    # (opcional) también log a fichero desde el hijo
    try:
        log_file = Path.cwd() / "incidencias_gui.log"
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setLevel(logging.INFO)
        fh.setFormatter(ui_fmt)
        root.addHandler(fh)
    except Exception:
        pass

    logging.info("--- Inicio de ejecución (subproceso) ---")
    try:
        logging.info("Aplicación: %s", app)
        logging.info("Modelos General: %s", modelos_gen)
        logging.info("Modelos Incidencia: %s", modelos_inci)
        logging.info("Excel: %s", fichero_path)

        cfg = cargar_settings()

        p = Path(fichero_path)
        url_dir = str(p.parent)   # path sin nombre
        fichero_name = p.name     # solo nombre

        # lista_campos
        raw_campos = cfg.get("DEFAULT", "lista_campos", fallback=None)
        if raw_campos:
            lista_campos = [c.strip() for c in raw_campos.split(",") if c.strip()]
        else:
            lista_campos = ["Número", "Descripción", "Notas de resolución"]

        thinking_incidencia = cfg.getboolean("DEFAULT", "thinking_incidencia", fallback=False)
        thinking_general = cfg.getboolean("DEFAULT", "thinking_general", fallback=False)
        try:
            temperature = cfg.getfloat("DEFAULT", "temperature")
        except Exception:
            temperature = 0.7

        # Prompt 1 (acepta ruta o texto literal)
        prompt1_value = cfg.get("DEFAULT", "prompt1_defecto", fallback="")
        prompt1 = _load_prompt(prompt1_value)
        logging.info("Prompt 1: %s", prompt1)

        if not inc or not hasattr(inc, "main_incidencias"):
            logging.error("Función main_incidencias no disponible en Incidencias.py")
            return

        logging.info(
            "Llamando a main_incidencias(url=%s, fichero=%s, lista_campos=%s, app=%s, "
            "thinking_incidencia=%s, thinking_general=%s, temperature=%s, "
            "lista_modelos_incidencias=%s, lista_modelos_general=%s)",
            url_dir, fichero_name, lista_campos, app,
            thinking_incidencia, thinking_general, temperature,
            modelos_inci, modelos_gen
        )

        # Llamada única solicitada
        inc.main_incidencias(
            url_dir,
            fichero_name,
            lista_campos,
            app=app,
            thinking_incidencia=thinking_incidencia,
            thinking_general=thinking_general,
            temperature=temperature,
            lista_modelos_incidencias=modelos_inci,
            lista_modelos_general=modelos_gen,
        )

        logging.info("main_incidencias completado correctamente.")
    except Exception as e:
        logging.exception("Error en main_incidencias: %s", e)
    finally:
        logging.info("--- Fin de ejecución (subproceso) ---")
'''

# ---------------------------
# Diálogo de edición de settings.ini
# ---------------------------
class SettingsDialog(tk.Toplevel):
    def __init__(self, master):
        super().__init__(master)
        self.title("Settings")
        self.geometry("720x520")
        self.vars: dict[tuple[str, str], tk.StringVar] = {}

        container = ttk.Frame(self)
        container.pack(fill="both", expand=True)

        canvas = tk.Canvas(container)
        vsb = ttk.Scrollbar(container, orient="vertical", command=canvas.yview)
        canvas.configure(yscrollcommand=vsb.set)
        vsb.pack(side="right", fill="y")
        canvas.pack(side="left", fill="both", expand=True)

        self.form = ttk.Frame(canvas)
        self.form.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=self.form, anchor="nw")

        ttk.Label(self.form, text="Parámetros de settings.ini",
                  font=("TkDefaultFont", 12, "bold")).grid(row=0, column=0, columnspan=3, pady=(10, 10))

        cfg = cargar_settings()
        row = 1
        sections = cfg.sections()
        if not sections:
            sections = ["DEFAULT"]
        for section in sections:
            s = section
            items = cfg.items(section) if section != "DEFAULT" else cfg.defaults().items()
            ttk.Separator(self.form, orient="horizontal").grid(row=row, column=0, columnspan=3, sticky="ew", pady=(10, 6))
            row += 1
            ttk.Label(self.form, text=f"[{s}]", font=("TkDefaultFont", 10, "bold")).grid(row=row, column=0, sticky="w")
            row += 1
            for key, val in items:
                ttk.Label(self.form, text=key).grid(row=row, column=0, sticky="w", padx=6, pady=4)
                var = tk.StringVar(value=val)
                entry = ttk.Entry(self.form, textvariable=var, width=80)
                entry.grid(row=row, column=1, sticky="ew", padx=6, pady=4)
                self.form.columnconfigure(1, weight=1)
                self.vars[(s, key)] = var
                row += 1

        btn_frame = ttk.Frame(self)
        btn_frame.pack(fill="x", padx=10, pady=10)
        ttk.Button(btn_frame, text="Guardar cambios", command=self._guardar).pack(side="right")

    def _guardar(self):
        cfg = configparser.ConfigParser()
        cfg["DEFAULT"] = {}
        for (section, key), var in self.vars.items():
            if section == "DEFAULT":
                cfg["DEFAULT"][key] = var.get()
            else:
                if section not in cfg:
                    cfg[section] = {}
                cfg[section][key] = var.get()
        with open(SETTINGS_PATH, "w", encoding="utf-8") as f:
            cfg.write(f)
        messagebox.showinfo("Settings", "Cambios guardados en settings.ini")
        self.destroy()

# ---------------------------
# Ventana principal
# ---------------------------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Incidencias IA - Visualización")
        self.geometry("1024x720")
        self.minsize(900, 600)

        # Menú
        self._build_menu()

        # Partes superior e inferior
        top = ttk.Frame(self, padding=10)
        top.pack(side="top", fill="x")
        bottom = ttk.Frame(self, padding=(10, 0, 10, 10))
        bottom.pack(side="top", fill="both", expand=True)

        # ------ Parte superior ------
        ttk.Label(top, text="Modelos General").grid(row=0, column=0, sticky="w", padx=6, pady=6)
        self.lb_modelos_general = tk.Listbox(top, selectmode="extended", exportselection=False, height=6)
        self.lb_modelos_general.grid(row=0, column=1, sticky="ew", padx=6, pady=6)
        self.lb_modelos_general.bind("<<ListboxSelect>>", self._on_select_modelos_general)

        ttk.Label(top, text="Modelos Incidencia").grid(row=1, column=0, sticky="w", padx=6, pady=6)
        self.lb_modelos_incidencia = tk.Listbox(top, selectmode="extended", exportselection=False, height=6)
        self.lb_modelos_incidencia.grid(row=1, column=1, sticky="ew", padx=6, pady=6)
        self.lb_modelos_incidencia.bind("<<ListboxSelect>>", self._on_select_modelos_incidencia)

        ttk.Label(top, text="Aplicación").grid(row=2, column=0, sticky="w", padx=6, pady=6)
        self.cb_aplicacion = ttk.Combobox(top, state="readonly")
        self.cb_aplicacion.grid(row=2, column=1, sticky="ew", padx=6, pady=6)

        self.btn_cargar = ttk.Button(top, text="Cargar Fichero", command=self._cargar_fichero)
        self.btn_cargar.grid(row=3, column=0, sticky="w", padx=6, pady=10)
        self.lbl_fichero = ttk.Label(top, text="(ningún fichero seleccionado)")
        self.lbl_fichero.grid(row=3, column=1, sticky="w", padx=6, pady=10)

        self.btn_ejecutar = ttk.Button(top, text="Ejecutar", command=self._ejecutar)
        self.btn_ejecutar.grid(row=4, column=0, sticky="w", padx=6, pady=10)

        top.columnconfigure(1, weight=1)

        # ------ Parte inferior (logging thread-safe multiproceso) ------
        self.txt_log = tk.Text(bottom, wrap="word", height=20)
        self.txt_log.pack(side="left", fill="both", expand=True)
        scroll = ttk.Scrollbar(bottom, orient="vertical", command=self.txt_log.yview)
        self.txt_log.configure(yscrollcommand=scroll.set)
        scroll.pack(side="right", fill="y")

        # Logging seguro para hilos (cola + poller) 
        self._setup_threadsafe_logging() 
        
        # Cargar listas iniciales
        self._cargar_modelos_y_aplicaciones()

        # Cierre limpio
        self.protocol("WM_DELETE_WINDOW", self._on_close)
        self._closing = False

    # -------------------- Menú --------------------
    def _build_menu(self):
        menubar = tk.Menu(self)

        m_archivo = tk.Menu(menubar, tearoff=False)
        m_archivo.add_command(label="Version", command=lambda: messagebox.showinfo("Versión", f"Versión de la aplicación: {__version__}"))
        menubar.add_cascade(label="Archivo", menu=m_archivo)

        m_settings = tk.Menu(menubar, tearoff=False)
        m_settings.add_command(label="Editar settings.ini…", command=self._abrir_settings)
        menubar.add_cascade(label="Settings", menu=m_settings)

        self.config(menu=menubar)

    # -------------------- Logging thread-safe --------------------
    def _setup_threadsafe_logging(self):
        # Quitar handlers previos de la GUI si re-ejecutamos sin reiniciar kernel
        root = logging.getLogger()
        for h in list(root.handlers):
            if getattr(h, "_from_gui", False):
                root.removeHandler(h)
                try:
                    h.close()
                except Exception:
                    pass

        # Usa la cola multiproceso para unificar logs de padre e hijo
        self._log_queue = queue.Queue()

        qh = logging.handlers.QueueHandler(self._log_queue) 
        qh.setLevel(logging.INFO) 
        qh._from_gui = True

        self._ui_formatter = logging.Formatter(
            "%(asctime)s - %(levelname)s - %(message)s", "%Y-%m-%d %H:%M:%S"
        )

        root.setLevel(logging.INFO)
        root.addHandler(qh)

        # (Opcional) fichero de log en el proceso padre
        try:
            log_file = Path.cwd() / "incidencias_gui.log"
            fh = logging.FileHandler(log_file, encoding="utf-8")
            fh.setLevel(logging.INFO)
            fh.setFormatter(logging.Formatter(
                "%(asctime)s - %(levelname)s - %(message)s", "%Y-%m-%d %H:%M:%S"
            ))
            fh._from_gui = True
            root.addHandler(fh)
        except Exception:
            pass

        self._closing = False
        self.after(100, self._drain_log_queue)

    def _drain_log_queue(self):
        if self._closing or not self.winfo_exists():
            return
        try:
            while True:
                record = self._log_queue.get_nowait()
                msg = self._ui_formatter.format(record)
                self.txt_log.configure(state="normal")
                self.txt_log.insert("end", msg + "\n")
                self.txt_log.see("end")
                self.txt_log.configure(state="disabled")
        except queue.Empty:
            pass
        finally:
            if not self._closing and self.winfo_exists():
                self.after(100, self._drain_log_queue)

    # -------------------- Carga inicial --------------------
    def _cargar_modelos_y_aplicaciones(self):
        # Modelos
        modelos = []
        try:
            if om and hasattr(om, "get_loaded_models"):
                modelos = om.get_loaded_models() or []
            else:
                logging.warning("get_loaded_models no disponible en ollama_manager. Lista vacía.")
        except Exception as e:
            logging.exception("Error al obtener modelos: %s", e)
            modelos = []
        self.lb_modelos_general.delete(0, "end")
        self.lb_modelos_incidencia.delete(0, "end")
        for m in modelos:
            self.lb_modelos_general.insert("end", m)
            self.lb_modelos_incidencia.insert("end", m)

        # Aplicaciones
        apps = []
        try:
            if inc and hasattr(inc, "get_lista_aplicaciones"):
                apps = inc.get_lista_aplicaciones() or []
            else:
                logging.warning("get_lista_aplicaciones no disponible en Incidencias. Lista vacía.")
        except Exception as e:
            logging.exception("Error al obtener aplicaciones: %s", e)
            apps = []
        self.cb_aplicacion["values"] = apps
        if apps:
            self.cb_aplicacion.current(0)

    # -------------------- Callbacks selección --------------------
    def _on_select_modelos_general(self, event=None):
        global modelos_general
        sel = [self.lb_modelos_general.get(i) for i in self.lb_modelos_general.curselection()]
        modelos_general = sel
        logging.info("Modelos General seleccionados: %s", modelos_general)

    def _on_select_modelos_incidencia(self, event=None):
        global modelos_incidencia
        sel = [self.lb_modelos_incidencia.get(i) for i in self.lb_modelos_incidencia.curselection()]
        modelos_incidencia = sel
        logging.info("Modelos Incidencia seleccionados: %s", modelos_incidencia)

    # -------------------- Archivo -> Cargar fichero --------------------
    def _cargar_fichero(self):
        global fichero_origen_path
        path = filedialog.askopenfilename(
            title="Selecciona el Excel de incidencias",
            filetypes=[("Excel", "*.xlsx *.xls"), ("Todos", "*.*")]
        )
        if path:
            fichero_origen_path = path
            self.lbl_fichero.configure(text=path)
            logging.info("Fichero seleccionado: %s", path)

    def _load_prompt(self, prompt_file: str) -> str:
        """Mantengo tu helper por si quieres usarla en el futuro (no imprescindible ya)."""
        prompt = ""
        if not os.path.exists(prompt_file):
            logging.error(f"El archivo {prompt_file} no existe.")
        else:
            try:
                with open(prompt_file, 'r', encoding='utf-8') as file:
                    prompt = file.read()
            except Exception as e:
                logging.exception("No se pudo leer el archivo de prompt: %s", e)
        return prompt

    # -------------------- Settings --------------------
    def _abrir_settings(self):
        _ = cargar_settings()  # asegura existencia del INI
        SettingsDialog(self)

    # -------------------- Ejecutar (lanza subproceso) --------------------
    def _ejecutar(self): 
        texto="" 
        app = self.cb_aplicacion.get().strip() if self.cb_aplicacion.get() else None 
        if not modelos_general: 
            texto='Selecciona al menos un modelo en "Modelos General".\n' 
        if not modelos_incidencia: 
            texto=f'{texto}Selecciona al menos un modelo en "Modelos General".\n' 
        if not app: 
            texto=f'{texto}Selecciona una aplicación.\n' 
        if not fichero_origen_path or not os.path.exists(fichero_origen_path): 
            texto=f'{texto}Selecciona un fichero Excel válido.\n' 
            
        if texto!="": 
            messagebox.showwarning("Validación", texto); 
            return 
        
        # Deshabilitar botón y lanzar hilo 
        self.btn_ejecutar.config(state="disabled") 
        logging.info("Lanzando ejecución en segundo plano…") 
        threading.Thread(target=self._run_pipeline, args=(app,), daemon=True).start()

    def _check_proc_done(self):
        if self._proc and not self._proc.is_alive():
            self.btn_ejecutar.config(state="normal")
        else:
            self.after(500, self._check_proc_done)

    def _run_pipeline(self, app: str): 
        try: 
            logging.info("--- Inicio de ejecución ---") 
            logging.info("Aplicación: %s", app) 
            logging.info("Modelos General: %s", modelos_general) 
            logging.info("Modelos Incidencia: %s", modelos_incidencia) 
            logging.info("Excel: %s", fichero_origen_path) 
            
            # Settings 
            cfg = cargar_settings() 
            
            # Separar dir y nombre de fichero 
            try: 
                p = Path(fichero_origen_path) 
                url_dir = str(p.parent) 
                
                # path sin nombre 
                fichero_name = p.name 
                
                # solo nombre 
            except Exception as e: 
                logging.exception("No se pudo descomponer la ruta del Excel: %s", e) 
                self.after(0, lambda: messagebox.showerror("Error", f"No se pudo descomponer la ruta del Excel.\n{e}")) 
                return 
        
            # lista_campos 
            raw_campos = cfg.get("DEFAULT", "lista_campos", fallback=None) 
            if raw_campos: 
                lista_campos = [c.strip() for c in raw_campos.split(",") if c.strip()] 
            else: 
                lista_campos = ["Número", "Descripción", "Notas de resolución"] 
        
            thinking_incidencia = cfg.getboolean("DEFAULT", "thinking_incidencia", fallback=False) 
            thinking_general = cfg.getboolean("DEFAULT", "thinking_general", fallback=False) 
            
            try: 
                temperature = cfg.getfloat("DEFAULT", "temperature") 
            except Exception: 
                temperature = 0.7 
                
            if not inc or not hasattr(inc, "main_incidencias"): 
                logging.error("Función main_incidencias no disponible en Incidencias.py") 
                self.after(0, lambda: messagebox.showerror("Error", "Función main_incidencias no disponible en Incidencias.py")) 
                return 
        
            logging.info( "Llamando a main_incidencias(url=%s, fichero=%s, lista_campos=%s, app=%s, " "thinking_incidencia=%s, thinking_general=%s, temperature=%s, " "lista_modelos_incidencias=%s, lista_modelos_general=%s)", url_dir, fichero_name, lista_campos, app, thinking_incidencia, thinking_general, temperature, modelos_incidencia, modelos_general ) 
            
            #Prompts por aplicaciones. 
            prompt1_file = cfg.get("DEFAULT", "prompt1_defecto", fallback="") 
            prompt1= self.load_prompt(prompt1_file) 
            logging.info(f"Prompt 1 por defecto: {prompt1}") 
            
            # Llamada única solicitada 
            inc.main_incidencias( url_dir, fichero_name, lista_campos, app=app, thinking_incidencia=thinking_incidencia, thinking_general=thinking_general, temperature=temperature, lista_modelos_incidencias=modelos_incidencia, lista_modelos_general=modelos_general) 
            logging.info("main_incidencias completado correctamente.") 
        except Exception as e: 
            logging.exception("Error en main_incidencias: %s", e) 
            self.after(0, lambda: messagebox.showerror("Error en ejecución", f"Ocurrió un error durante la ejecución.\n{e}")) 
        finally: 
            logging.info("--- Fin de ejecución ---") 
            self.after(0, lambda: self.btn_ejecutar.config(state="normal"))
    
    
    # -------------------- Cierre limpio --------------------
    def _on_close(self):
        # Parar poller y retirar handlers de la GUI 
        self._closing = True

        # Retirar handlers de logging conectados a la GUI
        root = logging.getLogger()
        for h in list(root.handlers):
            if getattr(h, "_from_gui", False):
                root.removeHandler(h)
                try:
                    h.close()
                except Exception:
                    pass
        try: 
            self.destroy() 
        except Exception: 
            pass


if __name__ == "__main__": 
    app = App() 
    app.mainloop()


In [ ]:
!pip install xlrd  --trusted-host pypi.org --trusted-host files.pythonhosted.org

   ---------------------------------------- 0.0/96.6 kB ? eta -:--:--
   ---- ----------------------------------- 10.2/96.6 kB ? eta -:--:--
   ---------------- ----------------------- 41.0/96.6 kB 653.6 kB/s eta 0:00:01
   ---------------------------------------- 96.6/96.6 kB 1.1 MB/s eta 0:00:00
